In [ ]:
import polars as pl

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config

RANDOM_SEED = 202605211526

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, with_imputation=False)

- ew5
- cc1
- cc2
- cc4_X
- cc6

In [ ]:
subset_df = dataset.question.filter(
    (pl.col("wave") == 3)
    | (pl.col("wave").is_in((4, 5)) & (pl.col("participant_type") == "repeating"))
)

subset_df = (
    (
        subset_df.join(
            (
                subset_df.filter(pl.col("wave") == 3)
                .select(
                    pl.col("item_id"), pl.len().over("item_id").alias("wave_3_count")
                )
                .unique()
            ),
            how="left",
            on="item_id",
        )
        .with_columns(satisfied_count=pl.len().over("item_id"))
        .filter(pl.col("satisfied_count") == 2 + pl.col("wave_3_count"))
        .select(pl.col("item_id"), pl.col("item_name"))
        .sort(by="item_id")
        .unique(maintain_order=True)
        .drop("item_id")
        .collect()
    )
    .to_series()
    .to_list()
)

In [ ]:
subset_df

In [ ]:
(
    dataset.participant.filter(
        wave_3=True,
        wave_4=True,
        wave_5=True,
    ).collect()
)